# NumPy 100 — Problems 61–70

Questions from [rougier/numpy-100](https://github.com/rougier/numpy-100) (MIT License). Solutions below are my own.

**Workflow:** try 10–15 min → docs → hints → official solution → rewrite from memory.

In [1]:
import numpy as np

---
### 61. Find the nearest value from a given value in an array (★★☆)

In [2]:
rng = np.random.default_rng(42)
Z = rng.integers(0, 100, 12)
z = 50
print(Z)

index = np.abs(Z - z).argmin()
print(index, Z[index])                  # the value closest to 50

# for a multi-dimensional array, use .flat to index the flattened view:
M = Z.reshape(3, 4)
print(M.flat[np.abs(M - z).argmin()])

[ 8 77 65 43 43 85  8 69 20  9 52 97]
10 52
52


**Concept:** same idea as problem 50: the distance to the target is `np.abs(Z - z)` and `argmin()` finds the smallest. `argmin` on a multi-dimensional array returns a *flat* index, so read the value with `M.flat[index]`. If two values are equally close, `argmin` returns the first.

**Needed help?** no  
**Revisit on:** not needed

---
### 62. Considering two arrays with shape (1,3) and (3,1), how to compute their sum using an iterator? (★★☆)

In [3]:
A = np.arange(3).reshape(3, 1)          # shape (3, 1)
B = np.arange(3).reshape(1, 3)          # shape (1, 3)

it = np.nditer([A, B, None])            # None -> let nditer allocate the output
for x, y, z in it:
    z[...] = x + y

result = it.operands[2]
print(result)
print(np.array_equal(result, A + B))    # True: same as simple broadcasting

[[0 1 2]
 [1 2 3]
 [2 3 4]]
True


**Concept:** `np.nditer` iterates over several arrays at once and broadcasts them together. Passing `None` creates an output array, and `z[...] = x + y` writes into the current element. It does what `A + B` does in one line, but shows how iterators work. In normal code just write `A + B`.

**Needed help?** solution  
**Revisit on:** Nov 15

---
### 63. Create an array class that has a name attribute (★★☆)

In [4]:
class NamedArray(np.ndarray):
    def __new__(cls, array, name="no name"):
        obj = np.asarray(array).view(cls)
        obj.name = name
        return obj

    def __array_finalize__(self, obj):
        if obj is None:
            return
        self.name = getattr(obj, "name", "no name")

Z = NamedArray(np.arange(10), "range_10")
print(Z.name)                           # range_10
print(Z[2:5].name)                      # range_10: slices keep the name
print((Z + 1).name)                     # range_10: results of operations keep it too
print(Z.sum())                          # normal array behavior still works

range_10
range_10
range_10
45


**Concept:** to subclass `np.ndarray` you create instances with `__new__` (using `.view(cls)`) rather than `__init__`. `__array_finalize__` runs whenever NumPy builds a new instance (slices, results of operations, copies), so copy your extra attributes there or they will be lost.

**Needed help?** solution  
**Revisit on:** Nov 15

---
### 64. Consider a given vector, how to add 1 to each element indexed by a second vector (be careful with repeated indices)? (★★★)

In [5]:
I = np.array([1, 1, 3, 3, 3, 5])        # note the repeated indices

# naive attempt: repeats are NOT counted, each index is only incremented once
Z = np.zeros(6)
Z[I] += 1
print(Z)                                # [0. 1. 0. 1. 0. 1.]

# correct way 1: count how often each index appears
Z = np.zeros(6)
Z += np.bincount(I, minlength=len(Z))
print(Z)                                # [0. 2. 0. 3. 0. 1.]

# correct way 2: unbuffered in-place add
Z = np.zeros(6)
np.add.at(Z, I, 1)
print(Z)                                # [0. 2. 0. 3. 0. 1.]

[0. 1. 0. 1. 0. 1.]
[0. 2. 0. 3. 0. 1.]
[0. 2. 0. 3. 0. 1.]


**Concept:** `Z[I] += 1` reads `Z[I]`, adds 1, then writes back, so repeated indices overwrite each other and only count once. `np.bincount(I, minlength=n)` counts occurrences of each index, and `np.add.at(Z, I, 1)` performs an unbuffered add that handles repeats correctly. `bincount` is usually much faster.

**Needed help?** solution  
**Revisit on:** Nov 15

---
### 65. How to accumulate elements of a vector (X) to an array (F) based on an index list (I)? (★★★)

In [6]:
X = [1, 2, 3, 4, 5, 6]
I = [1, 3, 9, 3, 4, 1]

F = np.bincount(I, weights=X)           # sum the X values for each index in I
print(F)                                # [0. 7. 0. 6. 5. 0. 0. 0. 0. 3.]

# check with np.add.at
G = np.zeros(10)
np.add.at(G, I, X)
print(np.array_equal(F, G))             # True

[0. 7. 0. 6. 5. 0. 0. 0. 0. 3.]
True


**Concept:** `np.bincount(I, weights=X)` adds up the weights `X` for every index value in `I`: index 1 appears with 1 and 6 (sum 7), index 3 with 2 and 4 (sum 6), and so on. The result is as long as `max(I) + 1`. It is a fast alternative to a loop or `np.add.at`.

**Needed help?** solution  
**Revisit on:** Nov 15

---
### 66. Considering a (w,h,3) image of (dtype=ubyte), compute the number of unique colors (★★☆)

In [7]:
rng = np.random.default_rng(42)
w, h = 16, 16
I = rng.integers(0, 4, (h, w, 3), dtype=np.ubyte)    # values 0-3, so at most 64 colors

# method 1: unique rows of the (pixels, 3) array
n1 = len(np.unique(I.reshape(-1, 3), axis=0))

# method 2: encode each RGB triple as one integer, then count unique integers
I32 = I.astype(np.uint32)               # widen first, otherwise ubyte overflows
F = I32[..., 0] * 256 * 256 + I32[..., 1] * 256 + I32[..., 2]
n2 = len(np.unique(F))

print(n1, n2, n1 == n2)

63 63 True


**Concept:** two ways: `np.unique(..., axis=0)` on the flattened `(pixels, 3)` array, or pack each RGB triple into a single integer `R*256² + G*256 + B` and count unique integers. Cast to a wider type (`uint32`) before doing the arithmetic, because `ubyte` values overflow past 255 (and in NumPy 2 even `* 256` raises an error).

**Needed help?** hint  
**Revisit on:** Nov 15

---
### 67. Considering a four dimensions array, how to get sum over the last two axis at once? (★★★)

In [8]:
rng = np.random.default_rng(42)
A = rng.integers(0, 10, (3, 4, 3, 4))

s1 = A.sum(axis=(-2, -1))               # sum over the last two axes at once
print(s1.shape)                         # (3, 4)

# older trick: flatten the last two axes and sum over one axis
s2 = A.reshape(A.shape[:-2] + (-1,)).sum(axis=-1)
print(np.array_equal(s1, s2))           # True

(3, 4)
True


**Concept:** `sum` (and `mean`, `max`, ...) accept a *tuple* of axes: `A.sum(axis=(-2, -1))` reduces both of the last two axes in one call. Negative axes count from the end. On old NumPy versions you had to reshape and sum over one merged axis instead.

**Needed help?** hint  
**Revisit on:** Nov 15

---
### 68. Considering a one-dimensional vector D, how to compute means of subsets of D using a vector S of same size describing subset  indices? (★★★)

In [9]:
rng = np.random.default_rng(42)
D = rng.uniform(0, 1, 100)
S = rng.integers(0, 10, 100)            # subset label (0-9) for every element of D

D_sums = np.bincount(S, weights=D)      # sum of D per subset
D_counts = np.bincount(S)               # size of each subset
D_means = D_sums / D_counts
print(D_means)

# check against a plain per-subset mean
check = np.array([D[S == k].mean() for k in range(10)])
print(np.allclose(D_means, check))      # True

# with pandas: pd.Series(D).groupby(S).mean()

[0.5300366  0.58193155 0.48533544 0.42422105 0.38529797 0.3690542
 0.51673882 0.61037507 0.48708886 0.50330849]
True


**Concept:** a grouped mean is `sum per group / count per group`, and `np.bincount` computes both: the counts directly, and the sums with `weights=D`. No loop needed. This is the NumPy equivalent of a pandas `groupby(...).mean()`. A group with no members would divide 0 by 0, giving `nan`.

**Needed help?** solution  
**Revisit on:** Nov 15

---
### 69. How to get the diagonal of a dot product? (★★★)

In [10]:
import timeit

rng = np.random.default_rng(42)
A = rng.uniform(0, 1, (5, 5))
B = rng.uniform(0, 1, (5, 5))

slow   = np.diag(np.dot(A, B))                  # computes the full 5x5 product first
faster = np.sum(A * B.T, axis=1)                # row i of A times column i of B
fastest = np.einsum("ij,ji->i", A, B)           # same, using Einstein summation
print(np.allclose(slow, faster), np.allclose(slow, fastest))   # True True

# timing on bigger matrices
A = rng.random((500, 500)); B = rng.random((500, 500))
for name, f in [("diag(dot)", lambda: np.diag(np.dot(A, B))),
                ("sum(A*B.T)", lambda: np.sum(A * B.T, axis=1)),
                ("einsum", lambda: np.einsum("ij,ji->i", A, B))]:
    print(f"{name:11} {timeit.timeit(f, number=5):.4f} s")

True True
diag(dot)   0.0377 s
sum(A*B.T)  0.0083 s
einsum      0.0015 s


**Concept:** only the diagonal entries are needed, and entry `i` is `sum_j A[i, j] * B[j, i]`. Computing that directly with `np.sum(A * B.T, axis=1)` or `np.einsum("ij,ji->i", A, B)` avoids the full matrix product, which does far more work (`O(n³)` vs `O(n²)`). Timings depend on your machine, but the gap grows with matrix size.

**Needed help?** solution  
**Revisit on:** Nov 15

---
### 70. Consider the vector [1, 2, 3, 4, 5], how to build a new vector with 3 consecutive zeros interleaved between each value? (★★★)

In [11]:
Z = np.array([1, 2, 3, 4, 5])
nz = 3                                  # zeros between consecutive values

Z0 = np.zeros(len(Z) + (len(Z) - 1) * nz, dtype=Z.dtype)
Z0[::nz + 1] = Z                        # every (nz+1)-th slot gets a value
print(Z0)

[1 0 0 0 2 0 0 0 3 0 0 0 4 0 0 0 5]


**Concept:** allocate a zero vector of length `len(Z) + (len(Z) - 1) * nz`, then assign the original values with a step slice `Z0[::nz + 1] = Z`. Step slicing writes every fourth position, leaving three zeros in between. Using `dtype=Z.dtype` keeps the result as integers.

**Needed help?** hint  
**Revisit on:** Nov 15